# 1. The lecture LP solved with SciPy's linprog

Lecture: *Week 2 - Linear Programming 1*, slides 21–25 (*Canonical Form* and *Solution via Python*).

**The problem.** A factory makes chairs $x$ and tables $y$ on one machine (slides 3 and 11–13):

maximise $P = 5x + 7y$ subject to $3x + 2y \le 144$ (machine hours), $x \ge 4y$ (at least four chairs per
table), $x, y \ge 0$.

In canonical form (slide 22): $c = (5, 7)$, $A = \begin{pmatrix} 3 & 2 \\ -1 & 4 \end{pmatrix}$, $b = (144, 0)$.

**How to run it.** *Run → Run All Cells*, or press Shift + Enter in each cell while you read. Nothing to install: SciPy is built into the browser's Python (the first import takes a few seconds).

**Expected result.** `x[1] = 41.14`, `x[2] = 10.29`, `fval = -277.71`. `linprog` minimises, so the printed
`fval` is **minus** the maximum profit: $P = £277.71$ at point B (slide 12).

*Original script: `5 Source Files/Python/linprog_lecture_eg.py`. Runs in the browser.*

Load `linprog`, SciPy's linear-programming solver. It always **minimises** $c^T x$ subject to `A_ub` $x \le$ `b_ub`, and by default every variable is $\ge 0$, so the non-negativity constraints need no rows of their own.

In [1]:
from scipy.optimize import linprog

The objective $P = 5x + 7y$. To **maximise** $P$ we ask `linprog` to **minimise** $-P = -5x - 7y$ (slide 24): the best point is the same, only the sign of the objective value flips.

In [2]:
# objective (row vector): linprog minimises, so to maximise we pass the negatives
c = [-5, -7]

The constraints, one row per inequality in the form *row* $\cdot (x, y) \le$ *number*; row $k$ of `A_ub` goes with entry $k$ of `b_ub`. $x \ge 4y$ is not in that form yet, so it is rewritten as $-x + 4y \le 0$ (slide 21).

In [3]:
# linear inequalities 
# (no. rows in A = no. rows in b)
A_ub = [[3, 2],    # 3x + 2y <= 144  (machine hours)
        [-1, 4]]   # -x + 4y <= 0    (x >= 4y: at least 4 chairs per table)
b_ub = [144, 0] 

Solve and print. `results.success` is `True` when an optimum was found; `results.x` holds the optimal values of the variables and `results.fun` the minimised objective (`fval`).

In [4]:
#  call the linprog function
results = linprog(c, A_ub, b_ub)

print(results.message)
# if results are a success then print out the optimal solution
if results.success:
    for k in range(len(results.x)):
        print(f"x[{k+1}] = {results.x[k]:.2f}")
    print(f"fval = {results.fun:.2f}")        


Optimization terminated successfully. (HiGHS Status 7: Optimal)
x[1] = 41.14
x[2] = 10.29
fval = -277.71


## What do you notice?

Put the answer back into the two constraints. Which of them hold with equality?

<details><summary>Answer</summary>

Both: $3(41.14) + 2(10.29) = 144$ and $41.14 \approx 4 \times 10.29$ (exactly $288/7 = 4 \times 72/7$). Both
constraints are **binding** at point B, as on slide 15, and `results.slack` is $(0, 0)$.
</details>

## Try it

1. Delete the second constraint (`A_ub = [[3, 2]]`, `b_ub = [144]`) and re-run.
2. Put both constraints back and change the objective to slide 18's $P = 6x + 4y$ (`c = [-6, -4]`).
3. Add the manager's demand $x \ge 60$ from slide 16 as a third row.

<details><summary>Answers</summary>

1. `x[1] = 0.00`, `x[2] = 72.00`, `fval = -504.00`: point B of slide 8, all tables and no chairs.
2. `fval = -288.00` ($P = 288$). The objective is parallel to $3x + 2y = 144$, so every point on that edge
   is optimal; `linprog` prints one of them ($x = 48$, $y = 0$).
3. Add the row `[-1, 0]` to `A_ub` and `-60` to `b_ub` ($x \ge 60$ is $-x \le -60$). `linprog` prints
   *The problem is infeasible*: $3 \cdot 60 = 180 > 144$.
</details>